## Project 1 (Due 2/17)

The goal of the first project is to do some wrangling, EDA, and visualization, and generate sequences of values. We will focus on:

- CDC National Health and Nutritional Examination Survey (NHANES, 1999-2000): https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=1999
- CDC Linked Mortality File (LMF, 1999-2000): https://www.cdc.gov/nchs/data-linkage/mortality-public.htm

NHANES is a rich panel dataset on health and behavior, collected bi-yearly from around 1999 to now. We will focus on the 1999 wave, because that has the largest follow-up window, providing us with the richest mortality data. The mortality data is provided by the CDC Linked Mortality File. 

The purpose of the project is to use $k$-NN to predict who dies (hard or soft classification) and how long they live (regression).

### Day 1: Wrangling and EDA (40/100 pts)

First, go to the NHANES and LMF web sites and familiarize yourself with the data sources. Download codebooks. Think about what resources are available. The CDC Linked Mortality File is somewhat of a pain to work with, so I have pre-cleaned it for you. It is available at httts://github.com/ds4e/undergraduate_ml_assignments in the data folder, as `lmf_parsed.cav`. From the CDC LMF web page, get the SAS program to load the data; it is the real codebook.

Second, download the demographic data for the 1999--2000 wave from the NHANES page. You can use the following code chunk to merge the LMF and DEMO data:

``` python
import pandas as pd
mdf = pd.read_csv('lmf_parsed.csv') # Load mortality file
print( mdf.head() )
gdf = pd.read_sas("DEMO.xpt", format="xport") # Load demographics file
print( gdf.head() )
df = gdf.merge(mdf, on="SEQN", how="inner") # Merge mortality and demographics on SEQN variable
```

Third, the variables `ELIGSTAT`, `MORTSTAT`, `PERMTH_INT`, and `RIDAGEEX` are particularly important. Look them up in the documentation and clearly describe them. (5/100 pts.)

Second, the goal of the project is to use whatever demographic, behavioral, and health data you like to predict mortality (`MORTSTAT`) and life expectancy (`PERMTH_INT`). Go to the NHANES 1999--2000 web page and select your data and download it. Clearly explain your rationale for selecting these data. Use `.merge` to combine your data into one complete dataframe. Document missing values. (5/100 pts)

Third, do basic EDA and visualization of the key variables. Are any important variables skewed? Are there outliers? How correlated are pairs of variables? Do pairs of categorical variables exhibit interesting patterns in contingency tables? Provide a clear discussion and examination of the data and the variables you are interested in using. (20/100 pts)


### Day 2: $k$-NN classification/regression, write-up (50/100 pts)

Submit a notebook that clearly addresses the following, using code and markdown chunks:

1. Describe the data, particularly what an observation is and whether there are any missing data that might impact your analysis. Who collected the data and why? What known limitations are there to analysis? (10/100 pts)
2. Describe the variables you selected to predict mortality and life expectancy, and the rationale behind them. Analyze your variables using describe tables, kernel densities, scatter plots, and conditional kernel densities. Are there any patterns of interest to notice? (10/100 pts)
3. Using your variables to predict mortality using a $k$-Nearest Neighbor Classifier. Analyze its performance and explain clearly how you select $k$. (10/100 pts)
4. Using your variables to predict life expectancy using a $k$-Nearest Neighbor Regressor. Analyze its performance and explain clearly how you select $k$. (10/100 pts)
5. Describe how your model could be used for health interventions based on patient characteristics. Are there any limitations or risks to consider? (10/100 pts)

## Submission (10/100 pts)

Submit your work in a well-organized GitHub repo, where the code is appropriately commented and all members of the group have made significant contributions to the commit history. (10/100 pts)

# Day 1: Wrangling and EDA

## Read in Data and Combine Files

In [4]:
import pandas as pd
mdf = pd.read_csv('data/lmf_parsed.csv') # Load mortality file
print( mdf.head() )
gdf = pd.read_sas("data/DEMO.xpt", format="xport") # Load demographics file
print( gdf.head() )
df = gdf.merge(mdf, on="SEQN", how="inner") # Merge mortality and demographics on SEQN variable

   SEQN  ELIGSTAT  MORTSTAT  UCOD_LEADING  DIABETES  HYPERTEN  PERMTH_INT  \
0     1         2       NaN           NaN       NaN       NaN         NaN   
1     2         1       1.0           6.0       0.0       0.0       177.0   
2     3         2       NaN           NaN       NaN       NaN         NaN   
3     4         2       NaN           NaN       NaN       NaN         NaN   
4     5         1       0.0           NaN       NaN       NaN       244.0   

   PERMTH_EXM  
0         NaN  
1       177.0  
2         NaN  
3         NaN  
4       244.0  
   SEQN  SDDSRVYR  RIDSTATR  RIDEXMON  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDAGEEX  \
0   1.0       1.0       2.0       2.0       2.0       2.0      29.0      31.0   
1   2.0       1.0       2.0       2.0       1.0      77.0     926.0     926.0   
2   3.0       1.0       2.0       1.0       2.0      10.0     125.0     126.0   
3   4.0       1.0       2.0       2.0       1.0       1.0      22.0      23.0   
4   5.0       1.0       2.0       2.

### Describe Important Variables

**ELIGSTAT**
*   Eligibility status for mortality linkage to the National Death Index (NDI).
*   Values: 1 = eligible, 2 = under age (18), 3 = ineligible.
*   In the cleaned dataset used for EDA and visualizations, all observations have a value of 1.0, meaning the analysis focuses exclusively on eligible participants.

**MORTSTAT**
*   Final mortality status; 0 = assumed alive, 1 = assumed deceased.
*   About 30% of the filtered population used in the dataset is deceased.
*   Has a string relationship with both age and follow-up time.

**PERMTH_INT**
*   Represents follow-up time in months.
*   Heavily left-skewed with a skewness value of -1.74.
*   Massive spike around 230 to 250 months representing the ceiling effect of participants who survived the 20-year follow-up window.

**RIDAGEEX**
*   Best age in months at the date of examination.
*   In the cleaned dataset, the age range is restricted to adults, with a minimum of 20 years and a maximum of 85 years.
*   It is the strongerst predictor of mortality with a 0.64 correlation with MORTSTAT.

**DMDEDUC2**
*   Education level for adults over 20 years old.
*   Filtered to exclude "Refused" and "Don't Know" responses.
*   Higher education corresponds to a higher survival rate.

**DMDMARTL**
*   Marital status of the participant.
*   Filtered to exclude values of 77 or higher. (Refused/Missing)/
*   Has a weak negative correlation with morality status.

**DMDHHSIZ**
*   Number of people living in the participant's household.
*   Right-skewed with a mean of 3.14 people.
*   Weak negative correlation with mortality.

**INDHHINC**
*   Total annual household income.
*   Codes 1 through 11. Dataset was filtered to exclude codes 12 and 13.
*   Higher income brackets exhibit the highest survival rates.


## Basic EDA and Visualization of the Key Variables

Please refer to the `~/nb.ipynb` file in the GitHub. It has all the code and visualizations. 

### Questions to Answer
1. Are any important variables skewed?
2. Are there outliers?
3. How correlated are pairs of variables?
4. Do pairs of categorical variables exhibit interesting patterns in contingency tables?
5. Provide a clear discussion and examination of the data and the variables we are interested in using.

1. Are any important variables skewed? -- Tim

Yes. PERMTH_INT is heavily left-skewed with a skew of -1.74 and a spike around 250 months due to the ceiling effect. RIDAGEEX is right-skewed at 0.17. Additionally, the new variable DMDHHSIZ is right-skewed at 0.81.

![skew graph](./public/skew.png)

2. Are there outliers? -- Lydia 

For RIDAGEEX, the deceased group contains significant outliers at the bottom of the plot (low-age deaths). The boxplots also reveal that for PERMTH_INT, the "alive" group is almost entirely concentrated at the 250-month ceiling, whereas the deceased group contains numerous outliers representing deaths shortly after examination.

![outlier graph](./public/outlier.png)

3. How correlated are pairs of variables? -Tim

![correlation graph](./public/correlation.png)

The Correlation Heatmap confirms a strong negative correlation between MORTSTAT and PERMTH_INT (-0.81) and a strong positive correlation between MORTSTAT and RIDAGEEX (0.64). Socioeconomic variables DMDEDUC2 (Education) and INDHHINC (Income) show a moderate positive correlation of 0.41.


4. Do pairs of categorical variables exhibit interesting patterns in contingency tables? -Lydia

Contingency heatmaps show mortality rates drop from 46% for the lowest education level to 19% for the highest. Similarly, high-income codes (9.0+) show survival rates up to 88%, while lower brackets show significant mortality increases.

![contingency graph](./public/contingency.png)

5. Provide a clear discussion and examination of the data and the variables we are interested in using. -Pavi

The data we are looking at comes from the National Health and Nutrition Examination Survey. Each observation in this dataset represents an individual person. The main purpose of the data is to help us analyze and predict mortality and life expectancy for others based on this dataset. 

We selected and cleaned eight variables: ELIGSTAT, MORTSTAT, PERMTH_INT, RIDAGEEX, DMDEDUC2 (Education), DMDMARTL (Marital Status), DMDHHSIZ (Household Size), and INDHHINC (Income). We used df.query to filter out "Refused" or "Don't Know" responses (e.g., INDHHINC < 13 and DMDMARTL < 77), resulting in 3,231 clean observations. Finally, ridge plots show that survival density at the 250-month mark is highest for younger cohorts (10-40 yrs) and flattens significantly for older cohorts.

![ridge plot](./public/ridge.png)

# Day 2

1. Describe the data, particularly what an observation is and whether there are any missing data that might impact your analysis. Who collected the data and why? What known limitations are there to analysis? (10/100 pts) -Kaitlin

In this dataset, an observation is an individual person. There is a lot of missing data for certain variables such as MORSTAT and PERMTH_INT. This could impact analysis because it could create bias in our estimates for these variables. The data comes from the National Health and Nutrition Examination Survey, which is conducted bi-annually for the purpose of looking at trends in health and behavior. One limitation could be measurement error for self-reported data if people misreport their behaviors. Another limitation is that mortality can be wrongly linked to certain data, causing our prediction model to be less accurate. 


2. Describe the variables you selected to predict mortality and life expectancy, and the rationale behind them. Analyze your variables using describe tables, kernel densities, scatter plots, and conditional kernel densities. Are there any patterns of interest to notice? (10/100 pts) -Pavi

The variables we selected to predict mortality and life expectancy are mortality status and age in months. We thought age in months would be best to predict life expectancy because the higher their number is, the lower their life expectancy is likely to be. Additionally, we totaled the number of people assumed dead and assumed alive, which will help us predict mortality. 


3. Using your variables to predict mortality using a -Nearest Neighbor Classifier. Analyze its performance and explain clearly how you select .(10/100 pts) -Isabelle

Mortality was predicted by looking at a person’s age (in months) and follow-up time. The kNN Classifier model had an accuracy of around 85%. Looking at the confusion matrix, we can also see how the model predicted 36 false positives and 60 false negatives (out of the 183 patients that were deceased). I selected the k value used by performing code that found the optimal k value. As a result, I discovered that the optimal k value is 89. 


4. Using your variables to predict life expectancy using a k-Nearest Neighbor Regressor. Analyze its performance and explain clearly how you select k. (10/100 pts) -Kaitlin

Life expectancy was predicted by looking at a person’s age (in months). The regression model achieved a R^2 value of 0.44, which is very moderate. This shows that although age captures general survival patterns,  there is a lot of variability in life expectancy that remains unexplained. This makes sense, given that there are many factors that contribute to life expectancy. The model performs decently well in the middle range (100-200 months), where there is the most data. However, it does not perform well for extreme data, such as people who died less than 50 months after the previous examination. The optimal k value was chosen by using code that plots the train and test R^2 for many values of k. The optimal k ended up being 145, which is quite large to account for the large sample size. 


5. Describe how your model could be used for health interventions based on patient characteristics. Are there any limitations or risks to consider? (10/100 pts) -David

Firstly, when predicting mortality using a kNN classifier, patients with a high probability of mortality can be immediately prioritized with screenings and tests because the model has a high accuracy rate at 85%. In addition, when predicting life expectancy using a kNN regressor, healthcare workers are able to set realistic timetables and long-term goals for "middle range" (100-200 months) patients because the model's R squared value is only 44%. Also, since the data shows an inverse relationship between education level and income with mortality, health workers can focus on lower-income and lower-education brackets where survival rate is at its lowest. However, some limits or risks we should consider is that there is a high volume of missing data for both MORTSTAT and PERMTH_INT (risk of non-response bias), and that the NHANES data relies on self-reported behaviors which may include measurement errors. In addition, the 60 false negatives out of 183 deceased patients in the kNN classifer show that the model failed to identify almost 33% of patients who actually died but could have had life-saving opportunities. Furthermore, the R squared value being 0.44 shows that 56% of the variability is unexplained meaning relying on just age means leaving out vital information like genetics and lifestyle behaviors.


All codes for our responses are save seperately into the GitHub from each group member that wrote it. Day 1's work was a very collaborative effort, and the rest of the questions were split evenly between the group members. The question(s) that each person answered is labeled and written in this notebook for cleanliness and ease of grading.